# End-to-end reconstruction comparison

This notebook summarizes reconstruction quality for the VQGAN and LlamaGen tokenizers across the evaluated datasets. It combines pixel/perceptual reconstruction metrics, distribution-level evaluator metrics, and qualitative original-versus-reconstruction examples.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Data and evaluation procedure

For each dataset, both models reconstructed the same ordered image stream. The ordering used by the scripts is deterministic: class folders are sorted first, and image files inside each class are sorted next.

The reconstruction scripts report PSNR, SSIM, and LPIPS as mean plus standard deviation across images. Separate evaluator runs compare each reconstruction NPZ against the matching reference NPZ and report FID, sFID, precision, recall, and Inception Score.


In [ ]:
CSV_PATH = Path('../end_to_end_results/metrics/end_to_end_metrics_numeric.csv')
df = pd.read_csv(CSV_PATH)

dataset_order = [
    'ImageNet',
    'ImageNet-V2',
    'ImageNet-Sketch',
    'ObjectNet',
    'RVL-CDIP',
    'BloodMNIST',
    'OrganAMNIST',
]
model_order = ['LlamaGen', 'VQGAN']

metrics = [
    ('psnr_mean', 'psnr_std', 'PSNR', 'higher is better'),
    ('ssim_mean', 'ssim_std', 'SSIM', 'higher is better'),
    ('lpips_mean', 'lpips_std', 'LPIPS', 'lower is better'),
    ('fid', None, 'FID', 'lower is better'),
    ('sfid', None, 'sFID', 'lower is better'),
    ('precision', None, 'Precision', 'higher is better'),
    ('recall', None, 'Recall', 'higher is better'),
    ('inception', None, 'Inception Score', 'higher is better'),
]

df


## Metric overview

The following plots compare both models across datasets. Error bars are shown for PSNR, SSIM, and LPIPS because those metrics are computed per image and summarized as mean plus standard deviation. Dashed horizontal lines mark each model's ImageNet baseline for quick comparison against out-of-domain datasets.


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(20, 24))
axes = axes.flatten()

x = np.arange(len(dataset_order))
bar_width = 0.34

model_offsets = {
    'LlamaGen': -bar_width / 2,
    'VQGAN': bar_width / 2,
}

model_colors = {
    'LlamaGen': 'C0',
    'VQGAN': 'C1',
}

for ax, (metric, std_metric, ylabel, direction) in zip(axes, metrics):
    for model in model_order:
        sub = (
            df[df['model'] == model]
            .set_index('dataset')
            .loc[dataset_order]
            .reset_index()
        )

        y = sub[metric].values
        yerr = sub[std_metric].values if std_metric is not None else None

        ax.bar(
            x + model_offsets[model],
            y,
            width=bar_width,
            label=model,
            color=model_colors[model],
            alpha=0.85,
            yerr=yerr,
            capsize=4 if yerr is not None else 0,
            error_kw=dict(linewidth=1, alpha=0.75),
        )

        baseline = df[
            (df['model'] == model) &
            (df['dataset'] == 'ImageNet')
        ][metric].iloc[0]

        ax.axhline(
            baseline,
            linestyle='--',
            linewidth=1.4,
            color=model_colors[model],
            alpha=0.75,
        )

        ax.annotate(
            f'{model} ImageNet',
            xy=(1.01, baseline),
            xycoords=('axes fraction', 'data'),
            color=model_colors[model],
            fontsize=9,
            va='center',
            ha='left',
            annotation_clip=False,
        )

    ax.set_title(f'{ylabel} ({direction})', fontsize=14, fontweight='bold', pad=14)
    ax.set_ylabel(ylabel, fontsize=11)

    ax.set_xticks(x)
    ax.set_xticklabels(dataset_order, rotation=28, ha='right', fontsize=10)

    ax.tick_params(axis='y', labelsize=10)
    ax.grid(axis='y', alpha=0.25)

    ax.margins(x=0.04)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()

fig.suptitle(
    'Reconstruction and Distribution Metrics Across Datasets
'
    "Dashed lines show each model's ImageNet baseline",
    fontsize=20,
    fontweight='bold',
    y=0.985,
)

fig.legend(
    handles,
    labels,
    loc='upper center',
    ncol=2,
    frameon=False,
    fontsize=13,
    bbox_to_anchor=(0.5, 0.935),
)

plt.subplots_adjust(
    top=0.88,
    bottom=0.06,
    left=0.08,
    right=0.86,
    hspace=0.50,
    wspace=0.38,
)

plt.savefig(
    'vq_end_to_end_reconstruction_comparison_metrics.png',
    dpi=300,
    bbox_inches='tight',
)

plt.show()


## Qualitative reconstruction samples

The sample folders contain matched triplets: `original.png`, `vqgan.png`, and `llamagen.png`. The originals were regenerated from the source datasets using the same deterministic ordering as the reconstruction scripts, so each original corresponds to the reconstruction index shown in the sample name.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

RESULTS_ROOT = Path('../end_to_end_results')
SIDE_BY_SIDE_ROOT = RESULTS_ROOT / 'side_by_side'

DATASETS = sorted(p.name for p in SIDE_BY_SIDE_ROOT.iterdir() if p.is_dir())
DATASETS


In [ ]:
def sample_dirs(dataset):
    dataset_dir = SIDE_BY_SIDE_ROOT / dataset
    return sorted(p for p in dataset_dir.iterdir() if p.is_dir() and p.name.startswith('sample_'))


def show_triplet(dataset, sample_idx=0, figsize=(12, 4)):
    sample_dir = sample_dirs(dataset)[sample_idx]
    paths = {
        'Original': sample_dir / 'original.png',
        'VQGAN': sample_dir / 'vqgan.png',
        'LlamaGen': sample_dir / 'llamagen.png',
    }

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    for ax, (title, path) in zip(axes, paths.items()):
        ax.imshow(Image.open(path))
        ax.set_title(title)
        ax.axis('off')

    fig.suptitle(f'{dataset} | {sample_dir.name}')
    fig.tight_layout()


def show_dataset_grid(dataset, sample_count=4, figsize_per_row=(12, 4)):
    dirs = sample_dirs(dataset)[:sample_count]
    fig, axes = plt.subplots(len(dirs), 3, figsize=(figsize_per_row[0], figsize_per_row[1] * len(dirs)))

    if len(dirs) == 1:
        axes = [axes]

    for row_axes, sample_dir in zip(axes, dirs):
        paths = [
            ('Original', sample_dir / 'original.png'),
            ('VQGAN', sample_dir / 'vqgan.png'),
            ('LlamaGen', sample_dir / 'llamagen.png'),
        ]
        for ax, (title, path) in zip(row_axes, paths):
            ax.imshow(Image.open(path))
            ax.set_title(f'{sample_dir.name} | {title}')
            ax.axis('off')

    fig.suptitle(dataset)
    fig.tight_layout()


## Dataset galleries

Each row shows one original image and the corresponding reconstructions from both models. These examples are meant for visual inspection only; the quantitative conclusions should come from the metric table and evaluator outputs above.


In [ ]:
show_dataset_grid('imagenet', sample_count=2)
show_dataset_grid('imagenet_v2', sample_count=2)
show_dataset_grid('imagenet_sketch', sample_count=2)
show_dataset_grid('objectnet', sample_count=2)
show_dataset_grid('organamnist', sample_count=2)
show_dataset_grid('rvl_cdip', sample_count=2)
show_dataset_grid('bloodmnist', sample_count=2)
